# Neural Networks and Deep Learning — Seminar 7: Attention and transformers
Prof. dr. Daniel Traian Pele — Bucharest University of Economic Studies (ASE); Romanian Academy, Institute for Economic Forecasting (IPE)

[Open in Colab](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/main/notebooks/L7_seminar.ipynb)

Seminar tasks on real data from `../data/clean` (see `../data/SOURCES.md`). Write your code in the empty cells under each task; keep the seed fixed so that your numbers can be compared.

In [ ]:
# Google Colab: copy the course repository, so that the data and the saved results are found
import os, subprocess, sys
if 'google.colab' in sys.modules and not os.getcwd().startswith('/content/NEURAL_BIZ'):
    if not os.path.isdir('/content/NEURAL_BIZ'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/danpele/NEURAL_BIZ.git', '/content/NEURAL_BIZ'], check=True)
    os.chdir('/content/NEURAL_BIZ/notebooks')

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
DATA = '../data/clean/'
SEED = 20261108
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
except ImportError:
    DEVICE = 'cpu'
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'axes.grid': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
print('device:', DEVICE)

## Setup

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tw = pd.read_csv(DATA + 'twitter_fin_sentiment.csv')            # 0 bearish, 1 bullish, 2 neutral
tr_all, te = tw[tw.split == 'train'].reset_index(drop=True), tw[tw.split == 'valid'].reset_index(drop=True)
tr, va = train_test_split(tr_all, test_size=0.10, stratify=tr_all.label, random_state=SEED)
tr, va = tr.reset_index(drop=True), va.reset_index(drop=True)
ytr, yva, yte = tr.label.values, va.label.values, te.label.values
tok = AutoTokenizer.from_pretrained('ProsusAI/finbert')
def enc(texts, n=64):
    e = tok(list(texts), padding='max_length', truncation=True, max_length=n, return_tensors='pt')
    return e['input_ids'], e['attention_mask']
Itr, Mtr = enc(tr.text); Iva, Mva = enc(va.text); Ite, Mte = enc(te.text)
print('train', len(tr), 'validation', len(va), 'test', len(te), ' vocabulary', tok.vocab_size)

**Task 1.** TF-IDF benchmark: compare unigrams with unigrams + bigrams (logistic regression, $C=10$, balanced class weights). Report macro-F1 and the recall of the bearish class on the test tweets.

In [ ]:
# your code here

**Task 2.** Attention by hand: for the first test tweet, take the input of FinBERT's first layer and the query and key projections of head 3; compute the attention matrix and check that it equals the one returned by the model.

In [ ]:
# your code here

**Task 3.** Why $\sqrt{d_k}$? With the same projections, compute the standard deviation of the raw scores $\mathbf q^\top\mathbf k$ and of the scaled scores, and the largest attention weight in each row with and without scaling.

In [ ]:
# your code here

**Task 4.** Zero-shot FinBERT: map its labels (positive, negative, neutral) to (bullish, bearish, neutral) and compute the confusion matrix on the test tweets. Which error is the most frequent?

In [ ]:
# your code here

**Task 5.** Long documents: split the three longest European Central Bank speeches into chunks of 510 tokens, score each chunk with zero-shot FinBERT, and report the share of negative chunks per speech.

In [ ]:
# your code here